# Transformers & Attention - Assignment 5

<a target="_blank" href="https://colab.research.google.com/github/sham-nlp/2026nlp-5-transformers/blob/main/05_transformer_assignment_student.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Name:** `عبد الرحمن قره بولاد`


**Date:** `5/3/2026`

---

## Instructions

Complete the code sections marked with `# YOUR CODE HERE`.

**Submission:** Submit this notebook with all cells executed and outputs visible.

---

## Part 1: Understanding Attention

Attention tells the model **which words to focus on**. For each word, we compute a score for every other word in the sentence.

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# A sentence about ancient Damascus trade:
# "Damascus traded silk spices gold"
# Each word is a 4-dimensional vector (embedding)
words = ["Damascus", "traded", "silk", "spices", "gold"]
x = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # Damascus (city)
    [0.0, 1.0, 0.0, 0.0],  # traded (action)
    [0.0, 0.0, 1.0, 0.0],  # silk (goods)
    [0.0, 0.0, 0.0, 1.0],  # spices (goods)
    [0.5, 0.5, 0.0, 0.0],  # gold (goods + valuable, similar to city)
])

print(f"Input shape: {x.shape}")  # (5 words, 4 dimensions)

Input shape: torch.Size([5, 4])


### Self-Attention Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q (Query)**: What am I looking for?
- **K (Key)**: What do I contain?  
- **V (Value)**: What information do I pass?

<details>
<summary>Hint 1</summary>

Use `x @ W` for matrix multiplication. For transpose, use `K.T` or `K.transpose(0, 1)`.
</details>

<details>
<summary>Hint 2</summary>

Step by step:
1. `scores = Q @ K.T` - compute raw attention scores
2. `scores = scores / sqrt(d_k)` - scale by sqrt of dimension
3. `weights = F.softmax(scores, dim=-1)` - normalize to probabilities
4. `output = weights @ V` - weighted sum of values
</details>

In [6]:
import math

d_k = x.shape[1]  # dimension = 4

# For simplicity, let Q = K = V = x
Q = x
K = x
V = x

# TODO: Implement self-attention
# Step 1: Compute attention scores (Q @ K.T)
scores = Q @ K.T # YOUR CODE HERE

# Step 2: Scale by sqrt(d_k)
scores = scores / math.sqrt(d_k) # YOUR CODE HERE

# Step 3: Apply softmax to get attention weights
attn_weights =F.softmax(scores, dim=-1) # YOUR CODE HERE

# Step 4: Multiply by V
output = attn_weights @ V # YOUR CODE HERE

print("Attention Weights (each row = attention for one word):")
print(torch.round(attn_weights * 100) / 100)
print()
print(f"Output shape: {output.shape}")  # Expected: torch.Size([5, 4])

Attention Weights (each row = attention for one word):
tensor([[0.2800, 0.1700, 0.1700, 0.1700, 0.2200],
        [0.1700, 0.2800, 0.1700, 0.1700, 0.2200],
        [0.1800, 0.1800, 0.2900, 0.1800, 0.1800],
        [0.1800, 0.1800, 0.1800, 0.2900, 0.1800],
        [0.2200, 0.2200, 0.1700, 0.1700, 0.2200]])

Output shape: torch.Size([5, 4])


In [7]:
# Visualize attention weights
print("Attention matrix visualization:")
print("      Damascus traded  silk spices   gold")
for i, word in enumerate(words):
    row = "  ".join([f"{attn_weights[i,j]:.2f}" for j in range(len(words))])
    print(f"{word:8s} {row}")

Attention matrix visualization:
      Damascus traded  silk spices   gold
Damascus 0.28  0.17  0.17  0.17  0.22
traded   0.17  0.28  0.17  0.17  0.22
silk     0.18  0.18  0.29  0.18  0.18
spices   0.18  0.18  0.18  0.29  0.18
gold     0.22  0.22  0.17  0.17  0.22


---

## Part 2: HuggingFace Introduction

**HuggingFace** is a platform for sharing ML models and datasets. Think of it like GitHub, but for AI.

Key components:
- `transformers`: Library for loading pre-trained models
- `datasets`: Library for loading datasets
- `Trainer`: High-level training API

Example usage:
```python
from transformers import AutoModel, AutoTokenizer
model = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
```

In [8]:
# Install required packages (run this first!)
!pip install -q transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00


In [9]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import numpy as np

---

## Part 3: Load Dataset

We use **SST-2** (Stanford Sentiment Treebank) - a sentiment analysis dataset with movie reviews.

Labels: `0` = negative, `1` = positive

In [10]:
# Load a small subset for fast training (Eid holiday assignment!)
dataset = load_dataset("stanfordnlp/sst2")

# Use only 500 training samples and 100 validation samples
train_data = dataset["train"].select(range(500))
val_data = dataset["validation"].select(range(100))

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print()
print("Example:")
print(f"  Text: {train_data[0]['sentence']}")
print(f"  Label: {train_data[0]['label']} ({'positive' if train_data[0]['label'] == 1 else 'negative'})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Training samples: 500
Validation samples: 100

Example:
  Text: hide new secretions from the parental units 
  Label: 0 (negative)


---

## Part 4: Tokenization

Convert text to numbers that the model can understand.

In [11]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Test tokenizer
text = "I love this movie!"
tokens = tokenizer(text)
print(f"Text: {text}")
print(f"Input IDs: {tokens['input_ids']}")
print(f"Decoded back: {tokenizer.decode(tokens['input_ids'])}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Text: I love this movie!
Input IDs: [101, 1045, 2293, 2023, 3185, 999, 102]
Decoded back: [CLS] i love this movie! [SEP]


<details>
<summary>Hint 1</summary>

Use `tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)`

This returns a dictionary with `input_ids`, `attention_mask`, etc.
</details>

In [12]:
print(f"Decoded back: {tokenizer.decode(999)}")

Decoded back: !


In [14]:
train_data

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 500
})

In [15]:
def tokenize_function(examples):

    # TODO: Tokenize the "sentence" field
    # Use truncation=True, padding="max_length", max_length=128
    return tokenizer(examples['sentence'],truncation=True, padding="max_length", max_length=128)# YOUR CODE HERE

train_tokenized = train_data.map(tokenize_function, batched=True)
val_tokenized = val_data.map(tokenize_function, batched=True)

print(f"Tokenized training samples: {len(train_tokenized)}")
print(f"Columns: {train_tokenized.column_names}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenized training samples: 500
Columns: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


---

## Part 4b: Understanding Tokenization

In [20]:
# Exercise 1: See how subword tokenization works
# tokenizer.tokenize() returns the text pieces (not token IDs)

sentence = "The Damascene merchant sold spices along the Silk Road"

# TODO: Use tokenizer.tokenize() to split the sentence into pieces
tokens = tokenizer.tokenize(sentence)# YOUR CODE HERE

print(f"Sentence: {sentence}")
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Sentence: The Damascene merchant sold spices along the Silk Road
Tokens: ['the', 'dam', '##as', '##cene', 'merchant', 'sold', 'spices', 'along', 'the', 'silk', 'road']
Number of tokens: 11


<details>
<summary>Hint for Exercise 2</summary>

Both use `.tokenize(sentence)`:
- DistilBERT: `tokenizer.tokenize(sentence)`
- GPT-2: `gpt2_tokenizer.tokenize(sentence)`

Watch for the `##` prefix in WordPiece (BERT) vs `Ġ` prefix in BPE (GPT-2).
</details>

In [21]:
# Exercise 2: Compare BPE vs WordPiece tokenizers

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
sentence = "The Damascene merchant sold spices along the Silk Road"

# TODO: Tokenize with both tokenizers using .tokenize()
distilbert_tokens = tokenizer.tokenize(sentence) # YOUR CODE HERE
gpt2_tokens =gpt2_tokenizer.tokenize(sentence) # YOUR CODE HERE

print(f"Sentence: {sentence}\n")
print(f"DistilBERT (WordPiece): {distilbert_tokens}")
print(f"  Token count: {len(distilbert_tokens)}\n")
print(f"GPT-2 (BPE): {gpt2_tokens}")
print(f"  Token count: {len(gpt2_tokens)}\n")
print(f"Difference: {abs(len(distilbert_tokens) - len(gpt2_tokens))} tokens")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sentence: The Damascene merchant sold spices along the Silk Road

DistilBERT (WordPiece): ['the', 'dam', '##as', '##cene', 'merchant', 'sold', 'spices', 'along', 'the', 'silk', 'road']
  Token count: 11

GPT-2 (BPE): ['The', 'ĠDam', 'asc', 'ene', 'Ġmerchant', 'Ġsold', 'Ġspices', 'Ġalong', 'Ġthe', 'ĠSilk', 'ĠRoad']
  Token count: 11

Difference: 0 tokens


<details>
<summary>Hint for Exercise 3</summary>

Access these as attributes:
- `tokenizer.vocab_size` — integer
- `tokenizer.special_tokens_map` — dictionary
</details>

In [25]:
# Exercise 3: Vocabulary size and special tokens

# TODO: Get vocabulary sizes
distilbert_vocab = tokenizer.vocab_size # YOUR CODE HERE
gpt2_vocab = gpt2_tokenizer.vocab_size # YOUR CODE HERE

print(f"DistilBERT vocab size: {distilbert_vocab:,}")
print(f"GPT-2 vocab size: {gpt2_vocab:,}")

# TODO: Get special tokens map
special = tokenizer.special_tokens_map # YOUR CODE HERE

print(f"\nDistilBERT special tokens: {special}")

DistilBERT vocab size: 30,522
GPT-2 vocab size: 50,257

DistilBERT special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


---

## Part 5: Load Model

We use **DistilBERT** - a smaller, faster version of BERT.

In [26]:
# Load model for sequence classification (2 labels: positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(f"Model loaded: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased
Parameters: 66,955,010


---

## Part 6: Training

We use the `Trainer` class - it handles the training loop for us!

### Example with Different Dataset and Model

You can use the same pattern with other datasets and models:

```python
# Arabic sentiment analysis example
dataset = load_dataset("ar_sarcasm")  # Arabic sarcasm detection
model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv02",  # Arabic BERT
    num_labels=2
)
```

Popular alternatives:
- **Datasets**: `imdb`, `yelp_polarity`, `ag_news`, `emotion`
- **Models**: `bert-base-uncased`, `roberta-base`, `albert-base-v2`

<details>
<summary>Hint 1</summary>

Create `TrainingArguments` with:
- `output_dir="./results"`
- `num_train_epochs=2`
- `per_device_train_batch_size=16`
- `per_device_eval_batch_size=16`
- `eval_strategy="epoch"`
- `logging_steps=10`
</details>

<details>
<summary>Hint 2</summary>

Create `Trainer` with:
- `model=model`
- `args=training_args`
- `train_dataset=train_tokenized`
- `eval_dataset=val_tokenized`
- `tokenizer=tokenizer`
</details>

In [39]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",# YOUR CODE HERE
    num_train_epochs=2 , # YOUR CODE HERE
    per_device_train_batch_size=16 ,# YOUR CODE HERE
    per_device_eval_batch_size=16 , # YOUR CODE HERE
    eval_strategy="epoch" ,# YOUR CODE HERE
    logging_steps=10 ,# YOUR CODE HERE
    report_to="none"
)

# Create trainer
trainer = Trainer(
    model=model ,# YOUR CODE HERE
    args=training_args , # YOUR CODE HERE
    train_dataset=train_tokenized ,# YOUR CODE HERE
    eval_dataset=val_tokenized , # YOUR CODE HERE
    processing_class=tokenizer # YOUR CODE HERE
)

print("Trainer created!")

Trainer created!


In [40]:
# Train the model (this will take a few minutes)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.617221,0.491719
2,0.358222,0.349584


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=64, training_loss=0.5040881596505642, metrics={'train_runtime': 709.8764, 'train_samples_per_second': 1.409, 'train_steps_per_second': 0.09, 'total_flos': 33116849664000.0, 'train_loss': 0.5040881596505642, 'epoch': 2.0})

---

## Part 7: Test the Model

In [41]:
# Test on new sentences
test_sentences = [
    "This movie was amazing!",
    "Terrible film, waste of time.",
    "It was okay, nothing special.",
]

# Tokenize test sentences
test_inputs = tokenizer(test_sentences, truncation=True, padding=True, return_tensors="pt")

# Move to same device as model
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_inputs = {k: v.to(device) for k, v in test_inputs.items()}
model = model.to(device)

# Get predictions
model.eval()
with torch.no_grad():
    outputs = model(**test_inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)

print("Predictions:")
for sentence, pred in zip(test_sentences, predictions):
    label = "positive" if pred.item() == 1 else "negative"
    print(f"  '{sentence}' -> {label}")

Predictions:
  'This movie was amazing!' -> positive
  'Terrible film, waste of time.' -> negative
  'It was okay, nothing special.' -> negative
